In [ ]:
# ═══════════════════════════════════════════════════════════════════
# INCLUDE-50 — TRAINING ONLY (skip extraction, load from cache)
# ═══════════════════════════════════════════════════════════════════

# ── 0. INSTALL ───────────────────────────────────────────────────────
import subprocess

# Step 1: Fix numpy + scipy first
subprocess.run(['pip', 'install', 'numpy==1.26.4', '-q'], check=True)
subprocess.run(['pip', 'install', 'scipy', '--upgrade', '-q'], check=True)

# Step 2: Install mediapipe AFTER numpy is fixed
subprocess.run(['pip', 'install', 'mediapipe==0.10.21', '-q'], check=True)

# Step 3: Verify
import numpy as np
import tensorflow as tf
print('NumPy :', np.__version__)
print('TF    :', tf.__version__)
print('GPU   :', tf.config.list_physical_devices('GPU'))

# ── 1. IMPORTS ───────────────────────────────────────────────────────
import os, json, pickle, warnings, random, time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, Model, callbacks
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from pathlib import Path
from collections import deque, Counter
from dataclasses import dataclass, field
from typing import Optional

warnings.filterwarnings('ignore')
print('TF version :', tf.__version__)
print('GPU        :', tf.config.list_physical_devices('GPU'))

# ── 2. CONFIG ────────────────────────────────────────────────────────
# Cache is loaded from a saved Kaggle dataset input
# Add your include50-cache dataset in the Data tab on the right
CACHE_FILE     = '/kaggle/input/datasets/nishitsinghal07/embeddings-include-50/include50_landmark_cache.npz'
OUTPUT_DIR     = '/kaggle/working'
MODEL_PATH     = f'{OUTPUT_DIR}/include50_bilstm.keras'
SCALER_PATH    = f'{OUTPUT_DIR}/include50_scaler.pkl'
LABEL_MAP_PATH = f'{OUTPUT_DIR}/include50_label_map.json'
CONFIG_PATH    = f'{OUTPUT_DIR}/include50_config.json'

SEQ_LEN      = 60
LANDMARK_DIM = 201
HAND_DIM     = 126
POSE_DIM     = 75
NUM_CLASSES  = 262
UPPER_BODY   = [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,23,24,25,26,27,28,29,30]

BATCH_SIZE = 32
EPOCHS     = 80
LR         = 1e-3
VAL_SPLIT  = 0.15
TEST_SPLIT = 0.10
SEED       = 42

# label map hardcoded from your include50_label_map.json
LABEL_MAP_DICT = {
    "0":"actor","1":"adult","2":"afternoon","3":"alive","4":"alright",
    "5":"animal","6":"artist","7":"attack","8":"author","9":"baby",
    "10":"bad","11":"bag","12":"ball","13":"bank","14":"bathroom",
    "15":"beautiful","16":"bed","17":"bedroom","18":"bicycle","19":"big_large",
    "20":"bill","21":"bird","22":"black","23":"blind","24":"blue",
    "25":"boat","26":"book","27":"box","28":"boy","29":"brother",
    "30":"brown","31":"bus","32":"camera","33":"car","34":"card",
    "35":"cat","36":"cell_phone","37":"chair","38":"cheap","39":"child",
    "40":"city","41":"clean","42":"clock","43":"clothing","44":"cold",
    "45":"colour","46":"computer","47":"cool","48":"court","49":"cow",
    "50":"crowd","51":"curved","52":"daughter","53":"dead","54":"deaf",
    "55":"death","56":"deep","57":"dirty","58":"doctor","59":"dog",
    "60":"door","61":"dream","62":"dress","63":"dry","64":"election",
    "65":"energy","66":"evening","67":"exercise","68":"expensive","69":"fall",
    "70":"family","71":"famous","72":"fan","73":"fast","74":"father",
    "75":"female","76":"fish","77":"flat","78":"friday","79":"friend",
    "80":"gift","81":"girl","82":"god","83":"good","84":"good_afternoon",
    "85":"good_evening","86":"good_morning","87":"good_night","88":"grandfather",
    "89":"grandmother","90":"green","91":"grey","92":"ground","93":"gun",
    "94":"happy","95":"hard","96":"hat","97":"he","98":"healthy",
    "99":"heavy","100":"hello","101":"high","102":"horse","103":"hospital",
    "104":"hot","105":"hour","106":"house","107":"how_are_you","108":"husband",
    "109":"i","110":"india","111":"it","112":"job","113":"key",
    "114":"king","115":"kitchen","116":"lamp","117":"laptop","118":"lawyer",
    "119":"letter","120":"library","121":"light","122":"location","123":"lock",
    "124":"long","125":"loose","126":"loud","127":"low","128":"male",
    "129":"man","130":"manager","131":"market","132":"marriage","133":"mean",
    "134":"medicine","135":"minute","136":"monday","137":"money","138":"monsoon",
    "139":"month","140":"morning","141":"mother","142":"mouse","143":"narrow",
    "144":"neighbour","145":"new","146":"newspaper","147":"nice","148":"night",
    "149":"office","150":"old","151":"orange","152":"page","153":"paint",
    "154":"pant","155":"paper","156":"parent","157":"park","158":"patient",
    "159":"peace","160":"pen","161":"pencil","162":"photograph","163":"pink",
    "164":"plane","165":"player","166":"pleased","167":"pocket","168":"police",
    "169":"poor","170":"president","171":"price","172":"priest","173":"queen",
    "174":"quiet","175":"race_(ethnicity)","176":"radio","177":"red",
    "178":"religion","179":"reporter","180":"restaurant","181":"rich",
    "182":"ring","183":"sad","184":"saturday","185":"school","186":"science",
    "187":"screen","188":"season","189":"second","190":"secretary","191":"shallow",
    "192":"she","193":"shirt","194":"shoes","195":"short","196":"sick",
    "197":"sign","198":"sister","199":"skirt","200":"slow","201":"small_little",
    "202":"soap","203":"soft","204":"soldier","205":"son","206":"sport",
    "207":"spring","208":"store_or_shop","209":"street_or_road","210":"strong",
    "211":"student","212":"suit","213":"summer","214":"sunday","215":"t-shirt",
    "216":"table","217":"tall","218":"teacher","219":"team","220":"technology",
    "221":"telephone","222":"television","223":"temple","224":"thank_you",
    "225":"they","226":"thick","227":"thin","228":"thursday","229":"tight",
    "230":"time","231":"today","232":"tomorrow","233":"tool","234":"train",
    "235":"train_station","236":"train_ticket","237":"transportation","238":"truck",
    "239":"tuesday","240":"ugly","241":"university","242":"waiter","243":"war",
    "244":"warm","245":"we","246":"weak","247":"wednesday","248":"week",
    "249":"wet","250":"white","251":"wide","252":"wife","253":"window",
    "254":"winter","255":"woman","256":"year","257":"yellow","258":"yesterday",
    "259":"you","260":"you_(plural)","261":"young"
}
label_map = {int(k): v for k, v in LABEL_MAP_DICT.items()}

# Save label map and config to working dir
with open(LABEL_MAP_PATH, 'w') as f:
    json.dump(LABEL_MAP_DICT, f, indent=2)
with open(CONFIG_PATH, 'w') as f:
    json.dump(dict(SEQ_LEN=SEQ_LEN, LANDMARK_DIM=LANDMARK_DIM,
                   HAND_DIM=HAND_DIM, POSE_DIM=POSE_DIM,
                   NUM_CLASSES=NUM_CLASSES, UPPER_BODY=UPPER_BODY), f, indent=2)
print('Config ready.')

# ── 3. LOAD CACHE ────────────────────────────────────────────────────
print(f'\nLoading cache from {CACHE_FILE}...')
data = np.load(CACHE_FILE)
X    = data['X'].astype(np.float32)   # (N, 60, 201)
y    = data['y'].astype(np.int32)

counts = np.bincount(y)
print(f'X shape      : {X.shape}')
print(f'NUM_CLASSES  : {NUM_CLASSES}')
print(f'Samples/class: min={counts.min()}  mean={counts.mean():.1f}  max={counts.max()}')
print(f'NaN:{np.isnan(X).sum()}  Inf:{np.isinf(X).sum()}')

# ── 4. SPLIT & SCALE ─────────────────────────────────────────────────
N, T, D = X.shape
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=TEST_SPLIT, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=VAL_SPLIT/(1-TEST_SPLIT), stratify=y_tmp, random_state=SEED)
print(f'\nSplit — train:{len(X_train)}  val:{len(X_val)}  test:{len(X_test)}')

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train.reshape(len(X_train), T*D)).reshape(-1,T,D).astype(np.float32)
X_val   = scaler.transform(X_val.reshape(len(X_val), T*D)).reshape(-1,T,D).astype(np.float32)
X_test  = scaler.transform(X_test.reshape(len(X_test), T*D)).reshape(-1,T,D).astype(np.float32)

with open(SCALER_PATH, 'wb') as f: pickle.dump(scaler, f)
print('Scaler saved.')

# ── 5. BUILD MODEL ───────────────────────────────────────────────────
def build_model(seq_len, dim, num_classes):
    inp = layers.Input(shape=(seq_len, dim))
    x   = layers.LayerNormalization()(inp)
    x   = layers.TimeDistributed(layers.Dense(256, activation='relu'))(x)
    x   = layers.Dropout(0.3)(x)
    x   = layers.Bidirectional(layers.LSTM(256, return_sequences=True))(x)
    x   = layers.Dropout(0.3)(x)
    x   = layers.Bidirectional(layers.LSTM(128))(x)
    x   = layers.Dropout(0.3)(x)
    x   = layers.Dense(128, activation='relu')(x)
    x   = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)
    return Model(inp, out)

model = build_model(SEQ_LEN, LANDMARK_DIM, NUM_CLASSES)
model.summary()
model.compile(
    optimizer=tf.keras.optimizers.Adam(LR),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# ── 6. TRAIN ─────────────────────────────────────────────────────────
counts_train = np.bincount(y_train, minlength=NUM_CLASSES)
class_weight = {i: len(y_train)/(NUM_CLASSES * max(c,1)) for i,c in enumerate(counts_train)}

cb_list = [
    callbacks.EarlyStopping(monitor='val_accuracy', patience=15,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                patience=7, min_lr=1e-6, verbose=1),
    callbacks.ModelCheckpoint(f'{OUTPUT_DIR}/include50_bilstm_best.keras',
                              monitor='val_accuracy', save_best_only=True, verbose=1)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight,
    callbacks=cb_list
)

model.save(MODEL_PATH)
print(f'\nModel saved -> {MODEL_PATH}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(history.history['loss'],         label='Train')
ax1.plot(history.history['val_loss'],     label='Val')
ax1.set_title('Loss'); ax1.legend()
ax2.plot(history.history['accuracy'],     label='Train')
ax2.plot(history.history['val_accuracy'], label='Val')
ax2.set_title('Accuracy'); ax2.legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/training_history.png', dpi=150)
plt.show()

# ── 7. TEST SET EVALUATION ───────────────────────────────────────────
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f'\nTest Accuracy : {acc*100:.2f}%')
print(f'Test Loss     : {loss:.4f}\n')

y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
print(f'{"Class":<30} {"Correct":>7} {"Total":>6} {"Acc":>6}')
print('-' * 55)
for i in range(NUM_CLASSES):
    mask = y_test == i
    if not mask.sum(): continue
    correct = (y_pred[mask] == i).sum()
    print(f'{label_map[i]:<30} {correct:>7} {mask.sum():>6} {correct/mask.sum()*100:>5.1f}%')

# ── 8. SENTENCE ASSEMBLER ────────────────────────────────────────────
ASSEMBLER_CONFIG = {
    'MIN_CONFIRM_FRAMES' : 3,
    'COOLDOWN_FRAMES'    : 4,
    'PAUSE_FRAMES'       : 8,
    'MIN_CONFIDENCE'     : 0.55,
    'MAX_SENTENCE_WORDS' : 10,
    'SESSION_TIMEOUT_SEC': 30,
}

@dataclass
class AssemblerState:
    sentence_words    : list  = field(default_factory=list)
    pred_buffer       : deque = field(default_factory=lambda: deque(maxlen=5))
    current_word      : Optional[str] = None
    current_word_count: int   = 0
    cooldown          : int   = 0
    pause_counter     : int   = 0
    last_activity     : float = field(default_factory=time.time)
    completed         : list  = field(default_factory=list)

class SentenceAssembler:
    def __init__(self, cfg=None):
        self.cfg = {**ASSEMBLER_CONFIG, **(cfg or {})}
        self.s   = AssemblerState()

    def reset(self): self.s = AssemblerState()

    def flush(self):
        if not self.s.sentence_words: return None
        sentence = ' '.join(self.s.sentence_words)
        self.s.completed.append(sentence)
        self.s.sentence_words = []
        self.s.current_word = None
        self.s.current_word_count = 0
        self.s.pause_counter = 0
        return sentence

    def push(self, word, confidence):
        cfg, s = self.cfg, self.s
        if time.time() - s.last_activity > cfg['SESSION_TIMEOUT_SEC']: self.reset()
        s.last_activity = time.time()
        result = dict(confirmed_word=None,
                      current_sentence=' '.join(s.sentence_words),
                      completed=None, state='IDLE')
        if s.cooldown > 0:
            s.cooldown -= 1
            result['state'] = 'CONFIRMED'
            return result
        if confidence < cfg['MIN_CONFIDENCE']:
            s.pause_counter += 1
            s.current_word_count = 0
            s.current_word = None
            if s.pause_counter >= cfg['PAUSE_FRAMES'] and s.sentence_words:
                result['completed'] = self.flush()
                result['current_sentence'] = ''
                result['state'] = 'PAUSED'
            return result
        s.pause_counter = 0
        s.pred_buffer.append(word)
        smoothed = Counter(s.pred_buffer).most_common(1)[0][0]
        if smoothed == s.current_word: s.current_word_count += 1
        else: s.current_word, s.current_word_count = smoothed, 1
        result['state'] = 'DETECTING'
        if s.current_word_count >= cfg['MIN_CONFIRM_FRAMES']:
            if not s.sentence_words or s.sentence_words[-1] != smoothed:
                s.sentence_words.append(smoothed)
                result['confirmed_word'] = smoothed
            s.current_word_count = 0
            s.cooldown = cfg['COOLDOWN_FRAMES']
            result['state'] = 'CONFIRMED'
            if len(s.sentence_words) >= cfg['MAX_SENTENCE_WORDS']:
                result['completed'] = self.flush()
                result['current_sentence'] = ''
            else:
                result['current_sentence'] = ' '.join(s.sentence_words)
        return result

# ── 9. INFERENCE FUNCTION ────────────────────────────────────────────
def predict_word(landmark_sequence, top_k=3):
    arr  = np.array(landmark_sequence, dtype=np.float32)
    flat = scaler.transform(arr.reshape(1, SEQ_LEN * LANDMARK_DIM))
    X_in = flat.reshape(1, SEQ_LEN, LANDMARK_DIM).astype(np.float32)
    probs   = model.predict(X_in, verbose=0)[0]
    top_idx = np.argsort(probs)[::-1][:top_k]
    return {
        'word'      : label_map[int(top_idx[0])],
        'confidence': float(probs[top_idx[0]]),
        'top_k'     : [{'word': label_map[int(i)], 'confidence': float(probs[i])}
                       for i in top_idx]
    }

print('predict_word() ready.')
print('\nAll done! Download these files:')
for fname in ['include50_bilstm.keras', 'include50_bilstm_best.keras',
              'include50_scaler.pkl', 'include50_label_map.json', 'include50_config.json']:
    path = f'{OUTPUT_DIR}/{fname}'
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024
        print(f'  {fname:<45} {size:>8.1f} KB')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 57.2 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.1 requires scipy<1.17,>=1.8, but you have scipy 1.17.1 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
kaggle-environments 1.27.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.36.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
2026-02-27 19:45:05.767502: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been regi

NumPy : 1.26.4
TF    : 2.19.0
GPU   : []
TF version : 2.19.0
GPU        : []
Config ready.

Loading cache from /kaggle/input/datasets/nishitsinghal07/embeddings-include-50/include50_landmark_cache.npz...


2026-02-27 19:45:18.403499: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


X shape      : (4257, 60, 201)
NUM_CLASSES  : 262
Samples/class: min=4  mean=16.2  max=22
NaN:0  Inf:0

Split — train:3192  val:639  test:426
Scaler saved.


...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 60, 201)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization             │ (None, 60, 201)        │           402 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 60, 256)        │        51,712 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 60, 512)        │     1,050,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 60, 512)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 256)            │       656,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 262)            │        33,798 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,825,816 (6.96 MB)

 Trainable params: 1,825,816 (6.96 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/80
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 416ms/step - accuracy: 0.0038 - loss: 5.6248
Epoch 1: val_accuracy improved from -inf to 0.01408, saving model to /kaggle/working/include50_bilstm_best.keras
100/100 ━━━━━━━━━━━━━━━━━━━━ 55s 458ms/step - accuracy: 0.0039 - loss: 5.6244 - val_accuracy: 0.0141 - val_loss: 5.4787 - learning_rate: 0.0010
Epoch 2/80
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 417ms/step - accuracy: 0.0097 - loss: 5.3407
Epoch 2: val_accuracy improved from 0.01408 to 0.02191, saving model to /kaggle/working/include50_bilstm_best.keras
100/100 ━━━━━━━━━━━━━━━━━━━━ 44s 443ms/step - accuracy: 0.0097 - loss: 5.3399 - val_accuracy: 0.0219 - val_loss: 4.8119 - learning_rate: 0.0010
Epoch 3/80
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 418ms/step - accuracy: 0.0372 - loss: 4.7868
Epoch 3: val_accuracy improved from 0.02191 to 0.04538, saving model to /kaggle/working/include50_bilstm_best.keras
100/100 ━━━━━━━━━━━━━━━━━━━━ 44s 443ms/step - accuracy: 0.0372 - loss: 4.7859 - val_accuracy: 0.0454 - 